# 🎧 RL Music Recommendation (TRL Training on Colab)

In [ ]:
# Clean conflicting stuff
!pip uninstall -y sentence-transformers diffusers trl transformers peft accelerate datasets -q

# Install correct stack
!pip install -q \
trl==0.7.10 \
transformers==4.37.2 \
peft==0.7.1 \
accelerate==0.25.0 \
datasets==2.16.1 \
openenv-core[core] \
matplotlib pandas numpy

# Restart runtime after this!

In [ ]:
# Clone repo
!git clone https://huggingface.co/spaces/aniketgala/music_rl_env
%cd music_rl_env

In [ ]:
# Check GPU
import torch
print("GPU:", torch.cuda.is_available())

In [ ]:
# Run training
!python trl_train.py

In [ ]:
# Generate 3 TRL graphs after training
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

root = Path('.')
summary_path = root / 'trl_summary.json'
debug_path = root / 'trl_debug.json'

if not summary_path.exists():
    raise FileNotFoundError('trl_summary.json not found. Run training cell first.')

summary = json.loads(summary_path.read_text(encoding='utf-8'))

# Rewards history source (preferred: trl_debug.json episode_rewards)
rewards_history = []
if debug_path.exists():
    debug_payload = json.loads(debug_path.read_text(encoding='utf-8'))
    rewards_history = debug_payload.get('episode_rewards', []) or []

if not rewards_history:
    # fallback to a flat list so plots still render
    rewards_history = [float(summary.get('mean_reward', 0.0))] * int(summary.get('episodes', 1))

# 1) Reward curve
plt.figure(figsize=(8, 4.5))
plt.plot(rewards_history, label='Reward')
plt.plot(pd.Series(rewards_history).rolling(10).mean(), label='Moving Avg')
plt.title('TRL Training Reward Curve')
plt.xlabel('Episodes')
plt.ylabel('Reward')
plt.legend()
plt.tight_layout()
plt.savefig('trl_training_curve.png')
plt.show()

# 2) Stability curve (loss proxy)
plt.figure(figsize=(8, 4.5))
rolling = pd.Series(rewards_history).rolling(10).mean()
plt.plot(rolling, label='Smoothed Reward')
plt.title('TRL Learning Stability (Proxy for Loss)')
plt.xlabel('Episodes')
plt.ylabel('Smoothed Reward')
plt.legend()
plt.tight_layout()
plt.savefig('trl_stability_curve.png')
plt.show()

# 3) Baseline vs TRL bar chart
baseline_mean = float(summary.get('baseline_mean', 0.0))
trl_mean = float(summary.get('mean_reward', 0.0))

plt.figure(figsize=(6.5, 4.5))
labels = ['Random Baseline', 'TRL Agent']
values = [baseline_mean, trl_mean]
plt.bar(labels, values)
plt.title('Baseline vs TRL Performance')
plt.ylabel('Average Reward')
for i, v in enumerate(values):
    plt.text(i, v, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('trl_comparison_bar.png')
plt.show()

print('\n=== FINAL TRL RESULTS ===')
print(f'Baseline: {baseline_mean:.3f}')
print(f'TRL: {trl_mean:.3f}')
print(f'Improvement: {trl_mean - baseline_mean:.3f}')
print('Saved: trl_training_curve.png, trl_stability_curve.png, trl_comparison_bar.png')